# 🚀 OpenAgent Eval — Complete Colab Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenAgentHQ/openagent-eval/blob/main/examples/openagent_eval_colab_tutorial.ipynb)
[![PyPI Version](https://img.shields.io/pypi/v/openagent-eval?color=blue&logo=pypi&logoColor=white)](https://pypi.org/project/openagent-eval/)
[![GitHub Stars](https://img.shields.io/github/stars/OpenAgentHQ/openagent-eval?style=social)](https://github.com/OpenAgentHQ/openagent-eval/stargazers)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://github.com/OpenAgentHQ/openagent-eval/blob/main/LICENSE)

**The open-source evaluation framework for RAG systems and AI Agents — learn it end-to-end, right in your browser.**

---

## 👋 Welcome

[OpenAgent Eval](https://github.com/OpenAgentHQ/openagent-eval) brings **pytest-level simplicity** to
AI evaluation. It measures how well a Retrieval-Augmented Generation (RAG) system retrieves the right
context and generates faithful, relevant answers — with 18+ metrics, corpus auditing, failure
diagnosis, and synthetic test-data generation.

This notebook is a **zero-setup, click-*Run all* tutorial**. It runs **completely offline with the
built-in `mock` providers** — so **you do not need any API keys** to complete every core section.
When you are ready to point it at a real LLM, the clearly-marked **Optional** cells show you how.

## ⏱️ What to expect
- **Estimated time:** ~30 minutes
- **Prerequisites:** none — just a Google account (or any Jupyter runtime)
- **Cost:** free (no paid Colab features, no API keys required for the core walkthrough)
- **Runtime:** the whole notebook finishes in well under a minute of compute

## 🗺️ Table of contents
1. [Installation & environment setup](#sec1)
2. [Terminal basics for Colab users](#sec2)
3. [The `oaeval` CLI at a glance](#sec3)
4. [Initialize your first configuration](#sec4)
5. [Prepare sample data](#sec5)
6. [Run your first evaluation — *no API key needed*](#sec6)
7. [Corpus health audit](#sec7)
8. [Failure diagnosis (blame attribution)](#sec8)
9. [Synthetic test-data generation](#sec9)
10. [Comparing experiments](#sec10)
11. [SDK usage (the Python API)](#sec11)
12. [CI/CD gating](#sec12)
13. [Advanced — custom metrics](#sec13)
14. [Tips, tricks & troubleshooting](#sec14)
15. [Next steps & resources](#sec15)
16. [Feedback & credit](#sec16)
17. [Optional — using real API keys](#sec17)


<a id="sec1"></a>
## 1. Installation & environment setup

Install OpenAgent Eval straight from PyPI. In Colab/Jupyter the `%pip` magic installs into the
kernel that is actually running this notebook, which is exactly what we want.

> **Why also `pytest`?** In release `0.4.8` the `oaeval` command-line tool imports `pytest` at
> start-up (it powers the `oaeval test` CI/CD command). Installing it alongside keeps every CLI
> command working. It is a tiny, pure-Python package, so the install stays fast.

In [1]:
%pip install -q openagent-eval pytest

Note: you may need to restart the kernel to use updated packages.


Keep the tool's output tidy by turning its debug logging down to warnings. Setting this in
`os.environ` means it is inherited by every `!` shell command we run later, too.

In [2]:
import os

# Quiet the library's debug logs so the tutorial output stays readable.
os.environ["LOGURU_LEVEL"] = "WARNING"

print("Environment ready.")

Environment ready.


Confirm the version and run the built-in environment doctor. `oaeval doctor` checks your Python
version, the installed dependencies, and which provider API keys are visible in the environment
(all *Not set* here — and that is fine, we will use the offline `mock` provider).

In [3]:
!oaeval --version

openagent-eval 0.4.8


In [4]:
!oaeval doctor

OpenAgent Eval - Environment Check



             Environment Status              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Component      ┃ Status ┃ Details         ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Python         │ OK     │ v3.13.14        │
│ openagent-eval │ OK     │ v0.4.8          │
│ typer          │ OK     │ CLI framework   │
│ rich           │ OK     │ Terminal UI     │
│ pydantic       │ OK     │ Data validation │
│ yaml           │ OK     │ Configuration   │
│ loguru         │ OK     │ Logging         │
│ jinja2         │ OK     │ HTML templates  │
│ httpx          │ OK     │ HTTP client     │
└────────────────┴────────┴─────────────────┘
             API Key Availability              
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Provider   ┃ Environment Variable ┃ Status  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ OpenAI     │ OPENAI_API_KEY       │ Not set │
│ Gemini     │ GEMINI_API_KEY       │ Not set │
│ Anthropic  │ ANTHROPIC_API_KEY    │ Not set │
│ Groq       │ GROQ_

<a id="sec2"></a>
## 2. Terminal basics for Colab users

Notebook-first? No problem. A **notebook cell can also run terminal (shell) commands** — you just
prefix the line with `!`. This section is a quick primer so the rest of the tutorial feels natural.

- A **plain cell** runs **Python**.
- A cell line starting with **`!`** runs a **shell command** (like a terminal).
- A cell starting with **`%%writefile <name>`** *writes the rest of the cell to a file*.
- A line starting with **`%`** is a **magic** command (notebook helper), e.g. `%pip`.

Let's try the basics.

In [5]:
!pwd          # print the current working directory
!ls -la       # list files here (there may be none yet)

/content


total 108
drwxr-xr-x 2 root root   4096 Jul 23 22:01 .
drwxr-xr-x 7 root root   4096 Jul 23 22:01 ..
-rw-r--r-- 1 root root 100013 Jul 23 22:01 nb.ipynb


In [6]:
!mkdir -p my_project
!ls -la my_project   # a freshly created (empty) directory

total 8
drwxr-xr-x 2 root root 4096 Jul 23 22:01 .
drwxr-xr-x 3 root root 4096 Jul 23 22:01 ..


### Writing and reading files

There is no `nano`/`vim` text editor in a notebook, but you don't need one. Use the `%%writefile`
magic to create a file, then `!cat` (or Python) to read it back.

In [7]:
%%writefile my_project/hello.txt
Hello from OpenAgent Eval!
This file was written straight from a notebook cell.

Writing my_project/hello.txt


In [8]:
!cat my_project/hello.txt        # view the whole file
print("---")
!head -1 my_project/hello.txt     # view just the first line

Hello from OpenAgent Eval!
This file was written straight from a notebook cell.


---
Hello from OpenAgent Eval!


> **Saving files locally:** in Colab you can download any file from the file browser (📁 icon in
> the left sidebar) → right-click → *Download*. To keep files between sessions, mount Google Drive
> (shown in the [troubleshooting section](#sec14)).

<a id="sec3"></a>
## 3. The `oaeval` CLI at a glance

Everything in OpenAgent Eval is available through the `oaeval` command. Here is the full command map:

| Command | What it does |
|---|---|
| `oaeval init` | Create a configuration file (wizard or defaults) |
| `oaeval validate <config>` | Validate a config without running |
| `oaeval run <config>` | Run the evaluation pipeline |
| `oaeval report <id>` | View an evaluation report (`latest` for the newest) |
| `oaeval list` | List previous evaluation runs |
| `oaeval compare <a> <b>` | Compare two experiments side by side |
| `oaeval audit <corpus>` | Audit corpus health before you wire up RAG |
| `oaeval diagnose <report>` | Attribute failures to retrieval / generation / chunking |
| `oaeval synth` | Generate synthetic test cases from a corpus or text |
| `oaeval test <config> -t ...` | Threshold-gated run for CI/CD |
| `oaeval doctor` | Environment & dependency check |

Any command's options are one `--help` away:

In [9]:
!oaeval --help

                                                                                
 Usage: oaeval [OPTIONS] COMMAND [ARGS]...                                      
                                                                                
 Open-source CLI framework for evaluating RAG systems and AI Agents.            
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --version             -V        Show the application's version and exit.     │
│ --quiet               -q        Suppress non-essential output.               │
│ --json                          Output machine-readable JSON.                │
│ --no-color                      Disable color output.                        │
│ --verbose             -v        Enable verbose output.                       │
│ --install-completion            Install completion for the current shell.    │
│ --show-completion         

In [10]:
!oaeval run --help

                                                                                
 Usage: oaeval run [OPTIONS] [config_path]                                      
                                                                                
 Run evaluation pipeline with the specified configuration.                      
                                                                                
 Args:                                                                          
     config_path (str | None): Path to the YAML configuration file. If not      
 provided,                                                                      
         auto-discovery will search for standard config filenames in the        
 current directory.                                                             
         Defaults to None.                                                      
     output (str): Override the output report format (terminal, markdown, html, 
 json).                     

<a id="sec4"></a>
## 4. Initialize your first configuration

`oaeval init` scaffolds a `config.yaml`. Its default is an **interactive wizard**, which would block
a *Run all*, so we pass `--no-interactive` to write sensible defaults instead. (When you work locally
in a real terminal, `oaeval init --interactive` gives you the friendly wizard.)

In [11]:
!oaeval init --no-interactive --force
print("---")
!cat config.yaml


OK Configuration created: config.yaml

Next steps:
  1. Review the configuration file
  2. Run oaeval validate to check it
  3. Run oaeval run to start evaluation


---


# OpenAgent Eval Configuration
# See documentation for options: https://github.com/OpenAgentHQ/openagent-eval
#
# For a fully offline dry-run (no API keys / vector store required), set:
#   llm.provider: mock
#   retriever.provider: mock

dataset:
  path: data/questions.json
  # limit: 100

llm:
  provider: openai
  model: gpt-4o-mini
  temperature: 0.0

retriever:
  provider: chroma
  settings:
    collection_name: my_collection

metrics:
  retrieval:
    - context_precision
    - context_recall
    - mrr
  generation:
    - faithfulness
    - answer_relevancy
  performance:
    - latency
  cost:
    - token_count

report:
  output: terminal
  output_dir: ./reports


The generated file is a good tour of every knob. The key sections are:

- **`dataset`** — where your test cases live and their format.
- **`llm`** — the model that generates answers (`provider`, `model`, `temperature`).
- **`retriever`** — the vector store / search backend.
- **`metrics`** — which retrieval, generation, performance, and cost metrics to compute.
- **`report`** — output format(s) and where reports are saved.

Notice the header comment even tells you how to go **fully offline** — set `llm.provider: mock` and
`retriever.provider: mock`. That is exactly what we will do next, so this notebook needs **no API
keys**.

<a id="sec5"></a>
## 5. Prepare sample data

A dataset is a JSON list of test cases. Each item can carry:

- `question` — the user query
- `ground_truth` — the reference (expected) answer
- `context` — the reference context/passage the answer should be grounded in
- `metadata` — anything extra you want to keep

Let's write a tiny dataset with `%%writefile`.

In [12]:
%%writefile data.json
[
  {
    "question": "What is Retrieval-Augmented Generation?",
    "ground_truth": "RAG combines a retriever and a generator to produce grounded answers.",
    "context": "Retrieval-Augmented Generation (RAG) pairs a retriever that fetches documents with a generator that writes an answer grounded in them.",
    "metadata": {"id": 1, "topic": "rag"}
  },
  {
    "question": "What is a vector database?",
    "ground_truth": "A vector database stores embeddings for similarity search.",
    "context": "Vector databases store high-dimensional embeddings and support fast nearest-neighbour similarity search.",
    "metadata": {"id": 2, "topic": "infra"}
  },
  {
    "question": "Why evaluate a RAG system?",
    "ground_truth": "To measure retrieval and generation quality and catch regressions.",
    "context": "Evaluating RAG quantifies retrieval accuracy and answer faithfulness so you can iterate and avoid regressions.",
    "metadata": {"id": 3, "topic": "eval"}
  }
]

Writing data.json


In [13]:
import json

cases = json.load(open("data.json"))
print(f"Loaded {len(cases)} test cases.")
print("First question:", cases[0]["question"])

Loaded 3 test cases.
First question: What is Retrieval-Augmented Generation?


<a id="sec6"></a>
## 6. Run your first evaluation — *no API key needed* 🔑🚫

This is the **"No API Key?"** path the docs promise. We write a config that uses the built-in
**`mock` LLM** and **`mock` retriever**. These are deterministic, offline stand-ins that let the
*entire* pipeline run in CI or in Colab with **zero secrets**.

In [14]:
%%writefile config.yaml
# Fully offline evaluation — no API keys required.
dataset:
  path: data.json
  format: json

llm:
  provider: mock          # deterministic, offline — no network calls
  model: mock-model
  temperature: 0.0

retriever:
  provider: mock          # deterministic, offline retriever
  settings:
    collection_name: demo_collection

metrics:
  retrieval:
    - context_precision
    - context_recall
    - mrr
  generation:
    - faithfulness
    - answer_relevancy
  performance:
    - latency
  cost:
    - token_count

report:
  output: terminal
  output_dir: ./reports

Overwriting config.yaml


Validate the config first — this catches typos and missing files before a run.

In [15]:
!oaeval validate config.yaml

OpenAgent Eval - Configuration Validator
Config: config.yaml

1. Checking YAML syntax...
  OK YAML syntax valid

2. Validating configuration schema...
  OK Configuration schema valid

3. Checking API keys...
  OK All required API keys configured

4. Checking dataset...
  OK Dataset found: data.json
  Size: 980 B

5. Checking output directory...
  OK Output directory exists: ./reports

6. Checking provider configuration...
  LLM: mock (mock-model)
  Retriever: mock

7. Checking metrics...
  Configured: 7 metrics
    Retrieval: context_precision, context_recall, mrr
    Generation: faithfulness, answer_relevancy
    Performance: latency
    Cost: token_count

Summary:
PASSED Configuration is valid
WARNING 1 warning(s)
  - Created output directory: reports

Ready to run: oaeval run <config>


Now run the evaluation. The mock providers make this finish instantly.

In [16]:
!oaeval run config.yaml

OpenAgent Eval v0.4.8
Configuration: config.yaml

⠋ Loading configuration... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0:00:00

  Complete! ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

OK Evaluation complete!
Items: 3 | Errors: 0
Report saved to: reports/a08f8fc3-06e4-4de3-ac11-eeb0858e7103.json


View the newest report with `oaeval report latest`.

In [17]:
!oaeval report latest

OpenAgent Eval - Report Viewer
Report: latest



╭──────────────────────────── Evaluation Complete ─────────────────────────────╮
│ OpenAgent Eval Report                                                        │
╰──────────────────────────────────────────────────────────────────────────────╯
      Summary      
┌─────────────┬───┐
│ Total Items │ 3 │
│ Successful  │ 3 │
│ Failed      │ 0 │
└─────────────┴───┘
           Metrics            
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Metric            ┃  Score ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ context_precision │ 0.0000 │
│ context_recall    │ 0.0000 │
│ mrr               │ 0.0000 │
│ faithfulness      │ 0.1250 │
│ answer_relevancy  │ 0.3333 │
│ latency           │ 1.0000 │
│ token_count       │ 0.9976 │
└───────────────────┴────────┘
                                 Sample Results                                 
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Question                                ┃ Metrics                        ┃
┡━━━╇━━━━━━━━━━━━━━━━━

### Reading the scores

Each metric is normalised to **0.0–1.0** (higher is better):

- **`context_precision` / `context_recall` / `mrr`** — *retrieval* quality: did we fetch the right
  documents, and how highly were they ranked?
- **`faithfulness`** — is the generated answer supported by the retrieved context (no hallucination)?
- **`answer_relevancy`** — does the answer actually address the question?
- **`latency`** — a normalised speed score.
- **`token_count`** — a normalised cost signal.

> **Note on the mock numbers:** because the `mock` retriever returns placeholder documents rather
> than a real vector search, the *retrieval* metrics here are illustrative (they show the pipeline
> working, not a real system's quality). Point OpenAgent Eval at a real retriever + LLM (see the
> [Optional section](#sec17)) to get meaningful scores. What matters right now: **every stage ran,
> offline, with no keys.**

Every run is saved to `./reports/`. List them any time:

In [18]:
!oaeval list

OpenAgent Eval - Evaluation History

                            Recent Evaluations                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ ID                                   ┃ Date       ┃ Config    ┃ Status ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ a08f8fc3-06e4-4de3-ac11-eeb0858e7103 │ 2026-07-23 │ data.json │ OK     │
└──────────────────────────────────────┴────────────┴───────────┴────────┘

Showing 1 evaluations


<a id="sec7"></a>
## 7. Corpus health audit

Garbage in, garbage out: if your knowledge base has **contradictions, stale documents, duplicates,
or coverage gaps**, no amount of prompt tuning will save your RAG system. `oaeval audit` scans a
folder of documents *before* you connect it to RAG. It runs fully offline.

In [19]:
import os, textwrap
os.makedirs("corpus", exist_ok=True)

with open("corpus/rag.md", "w") as f:
    f.write(textwrap.dedent("""
        # Retrieval-Augmented Generation
        RAG combines a retriever and a generator. The retriever fetches relevant
        documents from a knowledge base, and the generator produces an answer
        grounded in those documents.
    """).strip())

with open("corpus/vectors.md", "w") as f:
    f.write(textwrap.dedent("""
        # Vector Databases
        A vector database stores embeddings and supports similarity search over
        high-dimensional vectors. Examples include Chroma, Qdrant, and FAISS.
    """).strip())

print("Wrote a 2-document corpus.")

Wrote a 2-document corpus.


In [20]:
!oaeval audit ./corpus/

OpenAgent Eval v0.4.8
Corpus: ./corpus/

  Audit complete! ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  20% 0:00:00

╭──────────────────────────── Corpus Health Score ─────────────────────────────╮
│ 100.0% — Healthy                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

Summary: Corpus audit complete: 2 documents analyzed across 4 checks. No issues 
found.

No issues found!


The **health score** summarises the corpus. The four checks are:

- **Contradiction** — documents that assert conflicting facts.
- **Staleness** — documents older than a freshness threshold (`--staleness-days`).
- **Duplicate** — near-identical documents (`--similarity-threshold`).
- **Coverage** — thematic gaps across your corpus.

Run a subset with `--checks`, e.g. `!oaeval audit ./corpus/ --checks contradiction,duplicate`.

<a id="sec8"></a>
## 8. Failure diagnosis (blame attribution)

When an evaluation scores poorly, *which component is at fault?* `oaeval diagnose` reads a saved
report and attributes each failure to one of three culprits:

- **Retrieval** — the wrong documents were fetched.
- **Generation** — the LLM hallucinated or misread the context.
- **Chunking** — documents were split badly (too small, overlapping, etc.).

Point it at the report file we just produced.

In [21]:
import glob, os

latest_report = max(glob.glob("reports/*.json"), key=os.path.getmtime)
print("Diagnosing:", latest_report)

Diagnosing: reports/a08f8fc3-06e4-4de3-ac11-eeb0858e7103.json


In [22]:
!oaeval diagnose {latest_report}

╭──────────────────────────── Component Diagnosis ─────────────────────────────╮
│ Diagnosis Report                                                             │
│ Items analyzed: 3                                                            │
│ Overall health: 0.0%                                                         │
╰──────────────────────────────────────────────────────────────────────────────╯



System Health: Unhealthy (0.0%)

          Blame Attribution           
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Component  ┃ Failures ┃ Percentage ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Retrieval  │        6 │      54.5% │
├────────────┼──────────┼────────────┤
│ Generation │        5 │      45.5% │
└────────────┴──────────┴────────────┘

          Failure Modes          
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Failure Mode          ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Low Context Relevance │     3 │
├───────────────────────┼───────┤
│ Missing Context       │     3 │
├───────────────────────┼───────┤
│ Hallucination         │     3 │
├───────────────────────┼───────┤
│ Off-Topic Answer      │     2 │
└───────────────────────┴───────┘

Chunking Issues:

  overlapping_chunks: Contexts 1 and 2 have high overlap (similarity=0.82), 
suggesting duplicate chunking.

  overlapping_chunks: Contexts 1 and 3 have high overlap (similarity=0.82), 
suggesting duplicate chunking.

  overla

The output groups **failure signals** and then lists **recommendations** tagged by component
(`[RETRIEVAL]`, `[GENERATION]`, `[CHUNKING]`). Use the tag to decide where to spend your next hour:
re-chunking, swapping the embedding model, or tightening the generation prompt.

<a id="sec9"></a>
## 9. Synthetic test-data generation

Don't have a labelled test set yet? `oaeval synth` can **generate question/answer test cases from
your corpus** — including adversarial ones (unanswerable, misleading, multi-hop, …).

Synthesis *requires a capable LLM* to write the Q&A pairs. The offline `mock` provider returns
placeholder text, so running it offline **completes successfully but produces 0 cases** — a useful
way to confirm the command works end-to-end before you plug in a real model.

In [23]:
!oaeval synth --text "Retrieval-Augmented Generation pairs a retriever with a generator to produce grounded answers." \
    --count 3 --llm-provider mock --llm-model mock-model --format json --output synth.json

Standard generation failed: Failed to parse LLM response (response_preview=[mock-answer] You are a test case generator for a RAG (Retrieval-Augmented Generation) evaluation system.

Given the following document chunk, generate 3 diverse question-answer pairs.
Each question s)
Saved 0 test cases to synth.json


To get **real** synthetic test cases, run the same command with a real provider (e.g.
`--llm-provider openai --llm-model gpt-4o-mini`) after setting an API key — see the
[Optional section](#sec17). For example:

```bash
!oaeval synth --corpus ./corpus/ --count 20 --adversarial \
    --llm-provider openai --llm-model gpt-4o-mini --output dataset.json
```

<a id="sec10"></a>
## 10. Comparing experiments

Iterating on a RAG system means running many evaluations and asking *"did that change help?"*.
`oaeval compare` puts two runs side by side and computes per-metric deltas plus an overall winner.

Let's create a second run (a copy of our config) so we have two experiments to compare.

In [24]:
!cp config.yaml config_v2.yaml
!oaeval run config_v2.yaml

OpenAgent Eval v0.4.8
Configuration: config_v2.yaml



  Complete! ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00

OK Evaluation complete!
Items: 3 | Errors: 0
Report saved to: reports/319a2d80-ae9a-46e4-8c0f-5d8de9f5e32d.json


In [25]:
import glob, os

# Grab the two most recent report IDs (filenames without the .json extension).
reports = sorted(glob.glob("reports/*.json"), key=os.path.getmtime)
baseline_id = os.path.basename(reports[-2])[:-5]
experiment_id = os.path.basename(reports[-1])[:-5]
print("baseline  :", baseline_id)
print("experiment:", experiment_id)

baseline  : a08f8fc3-06e4-4de3-ac11-eeb0858e7103
experiment: 319a2d80-ae9a-46e4-8c0f-5d8de9f5e32d


In [26]:
!oaeval compare {baseline_id} {experiment_id}

OpenAgent Eval - Experiment Comparison
Comparing: a08f8fc3-06e4-4de3-ac11-eeb0858e7103 vs 
319a2d80-ae9a-46e4-8c0f-5d8de9f5e32d

  Experiment Comparison Report

  Generated: 2026-07-23 20:02:12 UTC
  Baseline:  a08f8fc3-06e4-4de3-ac11-eeb0858e7103
  Experiment: 319a2d80-ae9a-46e4-8c0f-5d8de9f5e32d

METRIC COMPARISON
------------------------------------------------------------
  Metric                           Baseline Experiment      Delta
  ----------------------------------------------------------
  answer_relevancy                   0.3333     0.3333 =  +0.0000
  context_precision                  0.0000     0.0000 =  +0.0000
  context_recall                     0.0000     0.0000 =  +0.0000
  faithfulness                       0.1250     0.1250 =  +0.0000
  latency                            1.0000     1.0000 =  -0.0000
  mrr                                0.0000     0.0000 =  +0.0000
  token_count                        0.9976     0.9976 =  +0.0000

SUMMARY
-----------------------

Because both runs used identical (mock) settings, every delta is `+0.0000`. In real use you'd
change the model, retriever, chunk size, or prompt between runs and watch the deltas move — a
compact, quantitative answer to *"is this version better?"*.

<a id="sec11"></a>
## 11. SDK usage (the Python API)

Prefer code over the CLI? Everything is available programmatically. Use the SDK when you want to
embed evaluation inside a larger Python workflow, build a custom loop, or inspect results in memory.

`Engine.run` is an **async** method, so we `await` it (top-level `await` works in Colab/Jupyter).

In [27]:
from openagent_eval.core import Engine
from openagent_eval.config import load_config

dataset = [
    {
        "question": "What is RAG?",
        "ground_truth": "Retrieval-Augmented Generation.",
        "context": "RAG combines retrieval and generation to produce grounded answers.",
    },
    {
        "question": "What is a vector database?",
        "ground_truth": "A store for embeddings enabling similarity search.",
        "context": "Vector databases store embeddings and support similarity search.",
    },
]

config = load_config("config.yaml")   # our offline mock config
engine = Engine(config)

report = await engine.run(dataset)     # async → await
print("Report type:", type(report).__name__)

Report type: EvaluationReport


In [28]:
# report.summary is a plain dict — easy to log, assert on, or serialise.
summary = report.summary
print("Items evaluated:", summary["total_items"])
print()
print("Metric scores:")
for name, score in summary["metrics_summary"].items():
    print(f"  {name:20s} {score:.4f}")

Items evaluated: 2

Metric scores:
  context_precision    0.0000
  context_recall       0.0000
  mrr                  0.0000
  faithfulness         0.1429
  answer_relevancy     0.0000
  latency              1.0000
  token_count          0.9979


That `metrics_summary` dict is the programmatic entry point for dashboards, alerts, or the
threshold checks we build next.

<a id="sec12"></a>
## 12. CI/CD gating

The point of evaluation is to **stop regressions from shipping**. OpenAgent Eval is designed to run
in CI: define **thresholds** (e.g. *faithfulness must be ≥ 0.8*) and fail the build if they are not
met.

The CLI form is:

```bash
oaeval test config.yaml -t faithfulness:gte:0.8 -t answer_relevancy:gte:0.7
```

where each `-t` gate is `metric:operator:value` (operators: `gt`, `gte`, `lt`, `lte`, `eq`, `neq`).
The command exits **0** when every gate passes and **1** when any gate fails — exactly what a CI
runner needs.

Below we demonstrate the **gating logic directly on the metrics we computed**, so you can see how a
pass/fail decision is made. This mirrors what a CI gate does with your report's scores.

In [29]:
scores = report.summary["metrics_summary"]

# Define the gates you'd enforce in CI: (metric, operator, threshold).
gates = [
    ("answer_relevancy", ">=", 0.5),
    ("latency",          ">=", 0.5),
    ("faithfulness",     ">=", 0.9),   # deliberately strict to show a failure
]

import operator as _op
OPS = {">=": _op.ge, ">": _op.gt, "<=": _op.le, "<": _op.lt, "==": _op.eq}

all_passed = True
for metric, sym, threshold in gates:
    actual = scores.get(metric)
    ok = actual is not None and OPS[sym](actual, threshold)
    all_passed &= ok
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {metric:18s} {actual:.4f} {sym} {threshold}")

print()
print("CI would exit with code", 0 if all_passed else 1)

[FAIL] answer_relevancy   0.0000 >= 0.5
[PASS] latency            1.0000 >= 0.5
[FAIL] faithfulness       0.1429 >= 0.9

CI would exit with code 1


### Wiring it into GitHub Actions

A minimal workflow that gates every push/PR on evaluation quality:

```yaml
name: Evaluation Gate
on: [push, pull_request]
jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install openagent-eval pytest
      - run: oaeval test config.yaml -t faithfulness:gte:0.8 -t answer_relevancy:gte:0.7
```

If any threshold fails, `oaeval test` returns a non-zero exit code and the job goes red.

<a id="sec13"></a>
## 13. Advanced — custom metrics

Need a metric that isn't built in? Subclass `BaseMetric` and return a `MetricResult`. The interface
is intentionally tiny:

- `name` / `description` — identify the metric.
- `evaluate(self, **kwargs) -> MetricResult` — receives the evaluation data as keyword arguments
  (`answer`, `context`, `question`, `ground_truth`, …) and returns a score.
- `MetricResult(score, reason, metadata)` — the `score` **must be between 0.0 and 1.0**.

Here's a custom metric that scores answers by normalised length.

In [30]:
from openagent_eval.metrics.base import BaseMetric, MetricResult


class AnswerWordCountMetric(BaseMetric):
    name = "answer_word_count"
    description = "Normalised answer length (word count / 100, capped at 1.0)."

    def evaluate(self, **kwargs) -> MetricResult:
        answer = kwargs.get("answer", "") or ""
        word_count = len(answer.split())
        return MetricResult(
            score=min(word_count / 100, 1.0),
            reason=f"Answer has {word_count} words",
            metadata={"word_count": word_count},
        )


metric = AnswerWordCountMetric()
result = metric.evaluate(
    answer="Retrieval-Augmented Generation combines a retriever with a generator."
)
print("score   :", result.score)
print("reason  :", result.reason)
print("metadata:", result.metadata)

score   : 0.08
reason  : Answer has 8 words
metadata: {'word_count': 8}


The 18+ built-in metrics all implement this same `BaseMetric` interface — you can browse them in
the `openagent_eval.metrics` package. The full built-in registry:

In [31]:
from openagent_eval.metrics import METRIC_REGISTRY

print("Built-in metrics:")
for name in sorted(METRIC_REGISTRY):
    print(" -", name)

Built-in metrics:
 - answer_relevancy
 - bertscore
 - bleu
 - context_precision
 - context_recall
 - exact_match
 - f1_score
 - faithfulness
 - hallucination
 - hit_rate
 - latency
 - mrr
 - ndcg
 - precision_at_k
 - recall_at_k
 - rouge
 - semantic_similarity
 - token_count


<a id="sec14"></a>
## 14. Tips, tricks & troubleshooting

**Higher-quality faithfulness/relevancy scores.** These metrics use a small NLI (natural-language
inference) model when `transformers` is available, and gracefully fall back to a lexical-overlap
approximation when it isn't. For NLI-grade scoring, install the extra:

```python
%pip install -q transformers torch
```

**Use a real LLM locally & for free with Ollama.** On a machine with a GPU you can run a local model
and point the config at it (`llm.provider: ollama`). See the docs for the Ollama provider.

**Persist files across Colab sessions — mount Google Drive:**

```python
from google.colab import drive
drive.mount('/content/drive')
# now read/write under /content/drive/MyDrive/...
```

**Common issues**

| Symptom | Fix |
|---|---|
| `oaeval: command not found` | Re-run the install cell; restart the runtime if needed. |
| A command asks interactive questions and hangs | Use `oaeval init --no-interactive` (as we do here). |
| Retrieval metrics are all `0.0` | Expected with the `mock` retriever — plug in a real one. |
| `synth` produced 0 cases | The `mock` LLM can't write Q&A — use a real provider ([§17](#sec17)). |
| Provider errors about missing API keys | You're using a real provider — set the key ([§17](#sec17)) or switch back to `mock`. |

**Share this notebook.** *File → Save a copy in Drive*, then *Share*, or push your copy to GitHub and
add the Colab badge from the top of this notebook.

<a id="sec15"></a>
## 15. Next steps & resources

- 📖 **Documentation:** <https://openagenthq.github.io/openagent-eval/>
- 💻 **GitHub repository:** <https://github.com/OpenAgentHQ/openagent-eval>
- 🐛 **Issues:** <https://github.com/OpenAgentHQ/openagent-eval/issues>
- 💬 **Discussions:** <https://github.com/OpenAgentHQ/openagent-eval/discussions>
- 📝 **Changelog:** <https://github.com/OpenAgentHQ/openagent-eval/blob/main/CHANGELOG.md>
- 🤝 **Contributing:** <https://github.com/OpenAgentHQ/openagent-eval/blob/main/CONTRIBUTING.md>
- 📚 **More example notebooks:** [`examples/`](https://github.com/OpenAgentHQ/openagent-eval/tree/main/examples)
  — see `rag_evaluation_tutorial.ipynb` and `corpus_and_related_modules.ipynb` for deeper dives.

**Related tools** in the RAG-evaluation space: RAGAS, DeepEval, TruLens. OpenAgent Eval's focus is a
**local-first, CLI-friendly** workflow with corpus auditing and blame attribution built in.

<a id="sec16"></a>
## 16. Feedback & credit

Was this notebook helpful? ⭐ **[Star OpenAgent Eval on GitHub](https://github.com/OpenAgentHQ/openagent-eval)**
— it genuinely helps the project.

- 🐞 Found a rough edge? [Open an issue](https://github.com/OpenAgentHQ/openagent-eval/issues).
- 💡 Have an idea to improve this tutorial? PRs to [`examples/`](https://github.com/OpenAgentHQ/openagent-eval/tree/main/examples)
  are very welcome.

Thanks for learning OpenAgent Eval! 🎉

<a id="sec17"></a>
## 17. Optional — using real API keys 🔐

Everything above ran **offline** with the `mock` providers. To evaluate a *real* RAG system, point
OpenAgent Eval at a real LLM. **These cells are optional** — the notebook is complete without them,
and each is written to **skip gracefully when no key is present**, so *Run all* stays green.

> 🔒 **Never hard-code a key.** Use `getpass` (prompts you, hidden input) or Colab's **Secrets
> Manager** (the 🔑 icon in the left sidebar), and read it from the environment.

In [32]:
# OPTIONAL — set an API key for this session without hard-coding it.
# Skips silently if you don't enter one (just press Enter to skip).
import os, getpass

# Preferred in Colab: read from the Secrets Manager (🔑 icon) if you've stored one there.
try:
    from google.colab import userdata  # type: ignore
    _secret = userdata.get("OPENAI_API_KEY")
    if _secret:
        os.environ["OPENAI_API_KEY"] = _secret
except Exception:
    pass

# Fallback: prompt (hidden). Press Enter to skip and stay fully offline.
# Wrapped so a non-interactive "Run all" (no keyboard) skips instead of erroring.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        entered = getpass.getpass("OPENAI_API_KEY (press Enter to skip): ")
        if entered.strip():
            os.environ["OPENAI_API_KEY"] = entered.strip()
    except Exception:
        print("(No interactive input available — staying offline.)")

print("OpenAI key set:", bool(os.environ.get("OPENAI_API_KEY")))

(No interactive input available — staying offline.)
OpenAI key set: False


With a key set, swap `provider: mock` for a real provider in your config and run as before.
The next cell **only runs a real evaluation if a key is present** — otherwise it explains how to
enable it and does nothing (so it's safe in *Run all*).

In [33]:
# OPTIONAL — real-provider evaluation. Guarded: no key → no-op.
if os.environ.get("OPENAI_API_KEY"):
    real_config = """
dataset:
  path: data.json
  format: json
llm:
  provider: openai
  model: gpt-4o-mini
  api_key: ${OPENAI_API_KEY}
retriever:
  provider: mock            # swap for chroma/faiss/... when you have a real corpus
  settings:
    collection_name: demo
metrics:
  generation: [faithfulness, answer_relevancy]
  performance: [latency]
  cost: [token_count]
report:
  output: terminal
  output_dir: ./reports
"""
    with open("config_real.yaml", "w") as f:
        f.write(real_config)
    print("Wrote config_real.yaml — running against OpenAI...")
    import subprocess
    subprocess.run(["oaeval", "run", "config_real.yaml"], check=False)
else:
    print("No OPENAI_API_KEY set — skipping the real run.")
    print("Set a key in the cell above (or Colab Secrets) to try it, then re-run this cell.")

No OPENAI_API_KEY set — skipping the real run.
Set a key in the cell above (or Colab Secrets) to try it, then re-run this cell.


---

*You made it! 🎉 You installed OpenAgent Eval, ran a full evaluation offline, audited a corpus,
diagnosed failures, compared experiments, used the SDK, built a custom metric, and saw how to gate
CI — all without a single API key. Happy evaluating!*